# Scraper Trabalha Brasil — Google Colab + Google Drive

Roda o projeto [Python-WebScraping](https://github.com/mateusaffonso/Python-WebScraping) no Colab e salva tudo no Google Drive:

| Arquivo no Drive | Conteúdo |
|---|---|
| `IC_scraper/listings/AAAA-MM-DD/page_N.html` | páginas da listagem, uma pasta por dia de coleta |
| `IC_scraper/vagas.csv` | uma linha por vaga: dados da listagem + data de publicação, faixa salarial, descrição completa, datas de acompanhamento |

**Datas**
- `date_posted`: data de publicação informada pelo site.
- O site **não publica a data de fechamento** (`valid_through` é preenchido automaticamente, ≈ publicação + 1 ano). O fechamento é inferido: a etapa 6 revisita as vagas abertas e preenche `closed_detected_at` quando a página passa a dizer "A vaga foi encerrada". Cada vaga aberta é revisitada no máximo uma vez por dia, então a precisão é de cerca de 1 dia.

Tudo **retoma de onde parou**: se o Colab desconectar, é só rodar de novo.

**Coleta automática:** o Colab gratuito não roda sozinho em horário marcado. A coleta diária roda no GitHub Actions (`.github/workflows/coleta-diaria.yml`, a cada 3 horas) e grava neste mesmo `vagas.csv` do Drive — veja *Automatic daily collection* no README.
Com ela ativa, use este notebook **para ver os dados** (seção 8) e evite rodar as etapas 4–6 enquanto houver uma coleta automática em andamento (aba Actions do GitHub), para as duas coletas não escreverem o `vagas.csv` ao mesmo tempo.

## 1. Configuração

In [ ]:
PASTA_DRIVE  = "/content/drive/MyDrive/IC_scraper"  # onde os dados ficam no Drive
MAX_PAGINAS  = 20     # páginas NOVAS da listagem nesta execução (15 vagas cada); None = até o fim
MAX_VAGAS    = 300    # páginas de vaga visitadas nesta execução (datas e detalhes); None = todas
SLEEP_MEDIA  = 2.0    # pausa média entre requisições (s) — seja gentil com o site
SLEEP_DESVIO = 0.5
ENGINE       = "playwright"   # "playwright" (Firefox headless) ou "requests" (mais leve)

REPO_URL = "https://github.com/mateusaffonso/Python-WebScraping.git"
BRANCH   = "feat/coleta-automatica"   # troque para "main" depois que os PRs forem aceitos

## 2. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(PASTA_DRIVE, exist_ok=True)
print("Dados em:", PASTA_DRIVE)

## 3. Baixar o projeto e instalar dependências
Na primeira execução de cada sessão leva ~1–2 min (download do Firefox para o Playwright).

In [ ]:
%cd /content
!rm -rf Python-WebScraping
!git clone -q -b {BRANCH} {REPO_URL}
%cd /content/Python-WebScraping
!pip install -q playwright beautifulsoup4 typer requests
if ENGINE == "playwright":
    !playwright install --with-deps firefox > /dev/null 2>&1 && echo "Firefox OK"

## 4. Coletar as páginas da listagem
Os comandos rodam com `!python` (processo separado) porque a API síncrona do Playwright não funciona dentro do loop `asyncio` do Colab.

In [ ]:
CLI = f'python proj_scraper/run.py trabalha-brasil'
opts = f'--data-folder "{PASTA_DRIVE}" --sleep-mean {SLEEP_MEDIA} --sleep-std {SLEEP_DESVIO} --engine {ENGINE}'
max_pag = f"--max-pages {MAX_PAGINAS}" if MAX_PAGINAS else ""
!{CLI} get-pages {opts} {max_pag}

## 5. Coletar os detalhes de cada vaga (data de publicação, salário, descrição)

In [ ]:
max_vagas = f"--max-jobs {MAX_VAGAS}" if MAX_VAGAS else ""
!{CLI} get-details {opts} {max_vagas}

## 6. (Opcional) Revisitar vagas abertas para detectar as encerradas
Rode periodicamente (ex.: uma vez por dia) para estimar a data de fechamento. Cada vaga é revisitada no máximo uma vez por dia.

In [ ]:
!{CLI} get-details {opts} --recheck {max_vagas}

## 7. Ver os dados

In [ ]:
import pandas as pd
df = pd.read_csv(f"{PASTA_DRIVE}/vagas.csv", dtype={"job_id": str})
print(f"{len(df)} vagas | com detalhes: {df.details_fetched_at.notna().sum()} | encerradas detectadas: {df.closed_detected_at.notna().sum()}")
df[["job_id", "title", "company", "location", "salary", "date_posted", "first_seen", "last_seen", "closed_detected_at"]].head(20)

## 8. Acompanhar a coleta automática
Resumo por dia de coleta e o log da última execução do GitHub Actions.

In [ ]:
import glob, os
import pandas as pd
df = pd.read_csv(f"{PASTA_DRIVE}/vagas.csv", dtype=str)
resumo = pd.DataFrame({
    "vagas_novas": df.first_seen.value_counts(),
    "encerradas_detectadas": df.closed_detected_at.dropna().str[:10].value_counts(),
}).fillna(0).astype(int).sort_index()
resumo.index.name = "dia"
print(resumo.tail(15))

logs = sorted(glob.glob(f"{PASTA_DRIVE}/logs/*.log"))
if logs:
    print(f"\n--- último log: {os.path.basename(logs[-1])}")
    print("".join(open(logs[-1], encoding="utf-8").readlines()[-15:]))
else:
    print("\nNenhuma execução automática ainda.")